# Towards a GIS system

In [ ]:
#| default_exp database

In [ ]:
#| export
#| export
import numpy as np
import os
import random
import asyncio
import tempfile
import shutil
import sqlite3
from math import radians, cos, sin, sqrt, atan2
from collections import namedtuple
from dataclasses import dataclass, field
from typing import List, Optional, Callable, Iterator
from enum import Enum
from datetime import datetime
from importlib import resources
from pathlib import Path

In [ ]:
#| export
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx

In [ ]:
#| export
#| export
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, HexRegion, unique_windy_edge, HexChunk
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.cover import ChunkCover, TerraDemo
from HexMagic.geology import River, Watershed

### Result Types

Small named tuples used everywhere for consistent return values.

In [ ]:
#| export

LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

## 4. ChunkRef and Coordinate Helpers

### What's going on here 

The world is tiled into **chunks** — groups of hexes. Each chunk has a
position in "chunk-space" using cube coordinates (q, r, s where q+r+s=0).

To go from a chunk's local hex to its world position, we multiply the
chunk position by `spacing = 2 × rings` to get the chunk center, then
add the local offset. Going backwards (world → chunk) we integer-divide.

In [ ]:
#| export
@dataclass
class ChunkRef:
    """Reference to a chunk in the world - uses HexPosition for coordinates."""
    position: HexPosition  # Chunk's position in chunk-space (q+r+s=0)
    rings: int             # Hex rings per chunk

    @property
    def center_hex(self) -> HexPosition:
        """World hex coordinates of chunk center."""
        spacing = self.rings * 2
        return spacing * self.position

    @property
    def key(self) -> tuple[int, int, int]:
        """Hashable key for dict storage."""
        return (self.position.q, self.position.r, self.position.s)

    def __hash__(self):
        return hash(self.key)

    @classmethod
    def from_key(cls, key, chunk_rings: int) -> 'ChunkRef':
        if isinstance(key, tuple):
            q, r, s = key
        else:
            q, r, s = map(int, key.split(','))
        return cls(HexPosition(q, r, s), chunk_rings)


def chunk_to_world(chunk: ChunkRef, local_pos: HexPosition) -> HexPosition:
    """Map local hex position within chunk to world coordinates."""
    return chunk.center_hex + local_pos


def world_to_chunk(world_pos: HexPosition, chunk_rings: int) -> tuple[HexPosition, HexPosition]:
    """Map world position to (chunk_position, local_position_within_chunk)."""
    spacing = chunk_rings * 2

    chunk_q = world_pos.q // spacing
    chunk_r = world_pos.r // spacing
    chunk_s = -chunk_q - chunk_r

    chunk_pos = HexPosition(chunk_q, chunk_r, chunk_s)
    chunk_center = spacing * chunk_pos
    local_pos = world_pos - chunk_center

    return chunk_pos, local_pos

**Test:**

In [ ]:
# Quick sanity check
ref = ChunkRef(HexPosition(1, 0, -1), rings=3)
assert ref.center_hex == HexPosition(6, 0, -6)
assert ref.key == (1, 0, -1)

# Round-trip: world → chunk → world
world = HexPosition(7, 2, -9)
cpos, local = world_to_chunk(world, chunk_rings=3)
back = chunk_to_world(ChunkRef(cpos, 3), local)
assert back == world, f"Round-trip failed: {world} → {back}"
print("✓ ChunkRef round-trip OK")

##  Table Dataclasses — HexData

Each hex in the database stores its cube coordinates (q, r, s), which
chunk it belongs to, its elevation, and a modification timestamp for
temporal versioning.

In [ ]:
#| export
@dataclass
class HexData:
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    grid_index: int = 0
    elevation: float = 0.0
    plate_id: int = -1
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    distance_from_coast: Optional[float] = None
    watershed_id: Optional[int] = None
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    scale_level: int = 0        # 0=coarse, >0=zoomed scale factor
    modified: int = 0
    last_accessed: int = 0
    access_count: int = 0

In [ ]:
#| export
@dataclass
class HexWeather:
    """Weather data for a hex at a point in time."""
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    temperature: float = 0.0           # °C
    precipitation: float = 0.0         # mm/year
    humidity: Optional[float] = None   # %
    climate_pet: Optional[float] = None
    aridity_index: Optional[float] = None
    temp_seasonality: Optional[float] = None
    precip_seasonality: Optional[float] = None
    distance_from_coast: Optional[float] = None
    season: str = ""
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    scale_level: int = 0
    climate_name: str = ""
    wind_dir: Optional[float] = None
    modified: int = 0

##  Table Dataclasses — World, User, ChunkBorder, ChunkMeta

In [ ]:
#| export
@dataclass
class World:
    """Metadata for a stored ChunkCover."""
    id: int = None
    name: str = ""
    extras: str = ""
    created: int = 0
    cover_data: str = ""
    modified: int = 0


@dataclass
class User:
    username: str
    email: str
    password: str
    created: int
    sessionID: str
    activeWorld: int
    id: int = None


@dataclass
class ChunkBorder:
    """Track drainage across chunk boundaries."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0
    chunk_s: int = 0
    border_hex_q: int = 0
    border_hex_r: int = 0
    border_hex_s: int = 0
    downstream_chunk_q: int = 0
    downstream_chunk_r: int = 0
    downstream_chunk_s: int = 0
    flow_volume: float = 0.0


@dataclass
class ChunkMeta:
    """Metadata for a cached chunk — avoids dimension inference."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0
    chunk_s: int = 0
    scale_level: int = 0
    nRows: int = 0
    nCols: int = 0
    radius: float = 0.0
    hex_count: int = 0
    created: int = 0
    last_accessed: int = 0
    access_count: int = 0

## GeoStorage — Init, DB Path, Context Manager

In [ ]:
#| export
class GeoStorage:

    def __init__(self, custom_path=None):
        path = GeoStorage.get_db_path(custom_path)
        self.path = path
        self.createDB()

    def createDB(self):
        db = database(self.path)
        self.db = db

        db.create(World, pk='id', if_not_exists=True, transform=True)
        db.create(HexData, pk='id', if_not_exists=True, transform=True)
        db.create(User, pk='id', if_not_exists=True, transform=True)
        db.create(HexWeather, pk='id', if_not_exists=True, transform=True)
        db.create(ChunkBorder, pk='id', if_not_exists=True, transform=True)
        db.create(ChunkMeta, pk='id', if_not_exists=True, transform=True)

        # Indices
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_world ON hex_data(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_coords ON hex_data(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_grid ON hex_data(world_id, grid_index)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_chunk ON chunk_border(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_downstream ON chunk_border(world_id, downstream_chunk_q, downstream_chunk_r, downstream_chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_chunk ON hex_data(world_id, chunk_q, chunk_r, chunk_s, scale_level)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_world ON hex_weather(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON hex_weather(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_temporal ON hex_weather(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_season ON hex_weather(world_id, season)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_chunk ON hex_weather(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_scale ON hex_weather(world_id, scale_level)")
        db.execute("""CREATE UNIQUE INDEX IF NOT EXISTS idx_chunk_meta_lookup
                    ON chunk_meta(world_id, chunk_q, chunk_r, chunk_s, scale_level)""")

        self.weather = db.t.hex_weather
        self.users = db.t.user
        self.hexes = db.t.hex_data
        self.worlds = db.t.world
        self.borders = db.t.chunk_border
        self.chunk_meta = db.t.chunk_meta

    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)

In [ ]:
#| export
@patch
def reset_database(self:GeoStorage):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)
    if db_path.exists():
        db_path.unlink()
    self.createDB()

In [ ]:
#| export
@patch
def _latest_hex_subquery(self: GeoStorage, world_id: int, as_of: int = None) -> str:
    """SQL subquery for latest hex state, optionally at a point in time."""
    time_clause = f"AND modified <= {as_of}" if as_of else ""
    return f"""
        SELECT world_id, q, r, s, MAX(modified) as max_mod
        FROM hex_data
        WHERE world_id = {world_id} {time_clause}
        GROUP BY world_id, q, r, s
    """

In [ ]:
#| export
@patch
def query_hexes_latest(self: GeoStorage, world_id: int, as_of: int = None) -> list:
    """Get most recent version of each hex, optionally at a specific time."""
    subq = self._latest_hex_subquery(world_id, as_of)

    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest
            ON h.world_id = latest.world_id
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """)

    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    return rows

In [ ]:
#| export
@patch
def update_hex(self: GeoStorage, world_id: int, q: int, r: int, s: int,
               elevation: float = None, latitude: float = None, longitude: float = None,
               distance_from_coast: float = None, grid_index: int = None) -> SaveResult:
    """Append a new state for a hex (temporal update)."""
    now = int(datetime.now().timestamp())

    record = {
        'world_id': world_id,
        'q': q, 'r': r, 's': s,
        'modified': now
    }

    prev = self.query_hexes_in_radius(world_id, q, r, s, radius=0)
    if prev:
        prev = prev[0]
        record['grid_index'] = grid_index if grid_index is not None else GeoStorage._get(prev, 'grid_index')
        record['elevation'] = elevation if elevation is not None else GeoStorage._get(prev, 'elevation')
        record['latitude'] = latitude if latitude is not None else GeoStorage._get(prev, 'latitude')
        record['longitude'] = longitude if longitude is not None else GeoStorage._get(prev, 'longitude')
        record['distance_from_coast'] = distance_from_coast if distance_from_coast is not None else GeoStorage._get(prev, 'distance_from_coast')
    else:
        record['grid_index'] = grid_index or 0
        record['elevation'] = elevation or 0.0
        record['latitude'] = latitude
        record['longitude'] = longitude
        record['distance_from_coast'] = distance_from_coast

    try:
        self.hexes.insert(record)
        return SaveResult(None, 'updated', f'hex ({q},{r},{s}) at {now}')
    except Exception as e:
        return SaveResult(None, 'error', str(e))

##  Hex Queries — query_hexes_in_radius

### Math note: hex distance

In cube coordinates, the distance between two hexes is:

    dist = max(|Δq|, |Δr|, |Δs|)

SQL can only filter on a rectangular bounding box (q and r ranges),
so we first get a superset from the DB, then filter by true hex distance
in Python.

In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)

    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest
            ON h.world_id = latest.world_id
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ])

    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]

    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)

    return result

##  World Persistence — save_cover / load_cover

In [ ]:
#| export
@patch
def save_cover(self: GeoStorage, cover: ChunkCover, name: str = "") -> SaveResult:
    """Save ChunkCover to database."""
    now = int(datetime.now().timestamp())

    try:
        with self.db.conn:
            if cover.ident < 0:
                cursor = self.db.execute("SELECT MAX(id) FROM world")
                max_id = cursor.fetchone()[0]
                cover.ident = (max_id or 0) + 1

            existing = self.db.execute(
                "SELECT id FROM world WHERE id = ?",
                [cover.ident]
            ).fetchone()

            encoded = cover.encode()
            record = {
                'id': cover.ident,
                'name': name or f"Cover_{cover.ident}",
                'cover_data': encoded,
                'modified': now
            }

            if existing:
                record['created'] = self.worlds[cover.ident]['created']
                self.worlds.update(record)
                status_msg = 'updated'
            else:
                record['created'] = now
                self.worlds.insert(record)
                status_msg = 'saved'

            return SaveResult(
                cover.ident,
                status_msg,
                f"ChunkCover {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
            )

    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_cover(self: GeoStorage, cover_id: int) -> LoadResult:
    """Load ChunkCover from database."""
    try:
        world = self.worlds[cover_id]
        if not world:
            return LoadResult(None, 'not_found', f'Cover {cover_id} not found')

        cover_data = GeoStorage._get(world, 'cover_data')
        if not cover_data:
            return LoadResult(None, 'error', 'No cover_data in world record')

        cover = ChunkCover.decode(cover_data)
        cover.ident = cover_id
        cover.db = self

        name = GeoStorage._get(world, 'name')
        return LoadResult(
            cover,
            'loaded',
            f"{name}: {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
        )

    except Exception as e:
        return LoadResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def save(self: ChunkCover, name: str = "") -> SaveResult:
    """Save this ChunkCover to its attached database."""
    if self.db is None:
        return SaveResult(None, 'error', 'No database attached to ChunkCover')

    result = self.db.save_cover(self, name=name)

    if result.status in ('saved', 'updated') and result.id is not None:
        self.ident = result.id

    return result


In [ ]:
#| export
@patch
def load_or_generate_chunk(self: GeoStorage,
                           cover: ChunkCover,
                           origin: int = None,
                           scale: int = 2,
                           method: str = 'bilinear',
                           octaves: int = 2,
                           persistence: float = 0.3,
                           detail_strength: float = 0.2,
                           force_regenerate: bool = False) -> LoadResult:
    """Load cached chunk or generate new one.

    Returns LoadResult(terrain, status, context) where:
    - status='cached' if loaded from DB
    - status='generated' if computed fresh
    """
    grid = cover.terrain.hexGrid
    if origin is None:
        origin = grid.middle

    origin_pos = grid.index_to_hexposition(origin)

    if not force_regenerate:
        cached = self._load_cached_chunk(cover.ident, origin_pos, scale)
        if cached.status == 'loaded':
            return LoadResult(cached.data, 'cached', cached.context)

    zoomed = cover.zoomChunkCombined(
        origin=origin, scale=scale, method=method,
        octaves=octaves, persistence=persistence,
        detail_strength=detail_strength
    )

    save_result = self._save_chunk_cache(cover.ident, origin_pos, scale, zoomed)

    if random.random() < 0.1:
        self.evict_lru_chunks(cover.ident, max_chunks=100)

    return LoadResult(zoomed, 'generated', f"{save_result.context}, scale={scale}")

## Chunk Cache 

In [ ]:
#| export
@patch
def invalidate_chunk_cache(self: GeoStorage, cover_id: int) -> SaveResult:
    """Delete all cached chunks and their metadata for a cover."""
    try:
        with self.db.conn:
            cursor = self.db.execute("""
                DELETE FROM hex_data
                WHERE world_id = ? AND scale_level > 0
            """, [cover_id])
            hex_count = cursor.rowcount

            cursor = self.db.execute("""
                DELETE FROM chunk_meta
                WHERE world_id = ?
            """, [cover_id])
            meta_count = cursor.rowcount

        return SaveResult(hex_count, 'invalidated',
                         f'{hex_count} cached hexes, {meta_count} metadata records deleted')

    except Exception as e:
        return SaveResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def _save_chunk_cache(self: GeoStorage,
                      cover_id: int,
                      origin_pos: HexPosition,
                      scale: int,
                      terrain: Terrain) -> SaveResult:
    """Save zoomed terrain as cached chunk, including metadata."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid

    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                if idx in grid.invalidRegion:
                    continue

                pos = grid.index_to_hexposition(idx)
                record = {
                    'world_id': cover_id,
                    'q': pos.q, 'r': pos.r, 's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'grid_index': idx,
                    'elevation': float(terrain.elevations[idx]),
                    'scale_level': scale,
                    'modified': now
                }
                self.hexes.insert(record)
                count += 1

            self.chunk_meta.insert({
                'world_id': cover_id,
                'chunk_q': origin_pos.q,
                'chunk_r': origin_pos.r,
                'chunk_s': origin_pos.s,
                'scale_level': scale,
                'nRows': grid.nRows,
                'nCols': grid.nCols,
                'radius': grid.radius,
                'hex_count': count,
                'created': now,
                'last_accessed': now,
                'access_count': 0
            })

            return SaveResult(count, 'saved', f'{count} hexes cached')

    except Exception as e:
        return SaveResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def _load_cached_chunk(self: GeoStorage,
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int) -> LoadResult:
    """Load cached chunk hexes from database."""
    try:
        cursor = self.db.execute("""
            SELECT h.* FROM hex_data h
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_data
                WHERE world_id = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest
                ON h.world_id = latest.world_id
                AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
                AND h.modified = latest.max_mod
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])

        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]

        if not rows:
            return LoadResult(None, 'not_found', 'No cached chunk')

        now = int(datetime.now().timestamp())
        self.db.execute("""
            UPDATE hex_data
            SET last_accessed = ?, access_count = access_count + 1
            WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
            AND scale_level = ?
        """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])

        terrain = self._reconstruct_terrain_from_rows(rows, cover_id)

        return LoadResult(terrain, 'loaded', f'{len(rows)} cached hexes')

    except Exception as e:
        return LoadResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def evict_lru_chunks(self: GeoStorage, cover_id: int,
                     max_chunks: int = 100) -> SaveResult:
    """Evict oldest chunks until under limit."""
    count = self.db.execute("""
        SELECT COUNT(DISTINCT chunk_q || ',' || chunk_r || ',' || chunk_s)
        FROM hex_data WHERE world_id = ? AND scale_level > 0
    """, [cover_id]).fetchone()[0]

    if count <= max_chunks:
        return SaveResult(0, 'ok', 'under limit')

    to_evict = count - max_chunks

    oldest = self.db.execute("""
        SELECT DISTINCT chunk_q, chunk_r, chunk_s, MIN(last_accessed) as oldest
        FROM hex_data
        WHERE world_id = ? AND scale_level > 0
        GROUP BY chunk_q, chunk_r, chunk_s
        ORDER BY oldest ASC
        LIMIT ?
    """, [cover_id, to_evict]).fetchall()

    deleted = 0
    with self.db.conn:
        for row in oldest:
            self.db.execute("""
                DELETE FROM hex_data
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level > 0
            """, [cover_id, row['chunk_q'], row['chunk_r'], row['chunk_s']])
            deleted += 1

    return SaveResult(deleted, 'evicted', f'{deleted} chunks removed')

In [ ]:
#| export
@patch
def _reconstruct_terrain_from_rows(self: GeoStorage, rows: list[dict], cover_id: int) -> Terrain:
    """Reconstruct Terrain from cached hex_data rows."""
    if not rows:
        return None

    qs = [r['q'] for r in rows]
    rs = [r['r'] for r in rows]

    world = self.worlds[cover_id]
    cover_data = GeoStorage._get(world, 'cover_data')
    if cover_data:
        original_cover = ChunkCover.decode(cover_data)
        base_radius = original_cover.terrain.hexGrid.radius
    else:
        base_radius = 10

    nRows = max(rs) - min(rs) + 1
    nCols = max(qs) - min(qs) + 1

    scale = rows[0].get('scale_level', 1)
    radius = base_radius / scale

    grid = HexGrid(nRows=nRows, nCols=nCols, radius=radius, style=StyleCSS("simple"))

    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}

    for row in rows:
        pos = HexPosition(row['q'], row['r'], row['s'])
        idx = grid.hexposition_to_index(pos)
        if 0 <= idx < len(terrain.elevations):
            terrain.elevations[idx] = row['elevation']

    return terrain

##  Weather Persistence — save / load / has_zoomed_weather

In [ ]:
#| export
@patch
def save_zoomed_weather(self: GeoStorage,
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        climate_name: str = "",
                        season: str = "annual") -> SaveResult:
    """Save weather from zoomed terrain, tagged by chunk origin."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())

    if 'temperature' not in terrain.fields or 'precipitation' not in terrain.fields:
        return SaveResult(None, 'error', 'Missing weather fields')

    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                if idx in invalid:
                    continue

                pos = grid.index_to_hexposition(idx)
                record = {
                    'world_id': cover_id,
                    'q': pos.q, 'r': pos.r, 's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'temperature': float(terrain.fields['temperature'][idx]),
                    'precipitation': float(terrain.fields['precipitation'][idx]),
                    'climate_name': climate_name,
                    'scale_level': scale,
                    'season': season,
                    'modified': now
                }
                for field in ['humidity', 'climate_pet', 'aridity_index']:
                    if field in terrain.fields:
                        record[field] = float(terrain.fields[field][idx])

                self.weather.insert(record)
                count += 1

            return SaveResult(count, 'saved',
                f'{count} weather records for chunk ({origin_pos.q},{origin_pos.r},{origin_pos.s})')

    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_zoomed_weather(self: GeoStorage,
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        season: str = "annual") -> LoadResult:
    """Load cached weather into zoomed terrain."""
    try:
        cursor = self.db.execute("""
            SELECT hw.* FROM hex_weather hw
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest
                ON hw.world_id = latest.world_id
                AND hw.q = latest.q AND hw.r = latest.r AND hw.s = latest.s
                AND hw.modified = latest.max_mod
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])

        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]

        if not rows:
            return LoadResult(terrain, 'not_found', 'No cached weather')

        n = len(terrain.elevations)
        for field in ['temperature', 'precipitation', 'humidity', 'climate_pet', 'aridity_index']:
            if field not in terrain.fields:
                terrain.fields[field] = np.zeros(n)

        grid = terrain.hexGrid
        filled = 0
        for row in rows:
            pos = HexPosition(row['q'], row['r'], row['s'])
            idx = grid.hexposition_to_index(pos)
            if 0 <= idx < n:
                terrain.fields['temperature'][idx] = row['temperature'] or 0.0
                terrain.fields['precipitation'][idx] = row['precipitation'] or 0.0
                if row.get('humidity') is not None:
                    terrain.fields['humidity'][idx] = row['humidity']
                if row.get('climate_pet') is not None:
                    terrain.fields['climate_pet'][idx] = row['climate_pet']
                if row.get('aridity_index') is not None:
                    terrain.fields['aridity_index'][idx] = row['aridity_index']
                filled += 1

        return LoadResult(terrain, 'loaded', f'{filled} weather records')

    except Exception as e:
        return LoadResult(terrain, 'error', str(e))


@patch
def has_zoomed_weather(self: GeoStorage,
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int,
                       season: str = "annual",
                       min_count: int = 1) -> bool:
    """Check if cached weather exists for this zoom."""
    try:
        cursor = self.db.execute("""
            SELECT COUNT(*) FROM hex_weather
            WHERE world_id = ? AND season = ?
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        count = cursor.fetchone()[0]
        return count >= min_count
    except Exception:
        return False

#| export
@patch
def zoom_cached(self: ChunkCover,
                origin: int = None,
                scale: int = 2,
                **kwargs) -> Terrain:
    """Zoom with database caching (requires db attached)."""
    if self.db is None:
        return self.zoomChunkCombined(origin=origin, scale=scale, **kwargs)

    result = self.db.load_or_generate_chunk(
        self, origin=origin, scale=scale, **kwargs
    )

    if result.status == 'cached':
        print(f"✓ Loaded from cache: {result.context}")
    else:
        print(f"⚙ Generated: {result.context}")

    return result.data

@patch
def zoom_with_full_data(self: ChunkCover,
                        origin: int = None,
                        scale: int = 2,
                        compute_weather: bool = True,
                        force_regenerate: bool = False) -> DrainageBasins:
    """Zoom chunk with weather and watersheds, using cache where possible.

    Returns DrainageBasins with terrain set to the zoomed terrain.
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for caching")

    grid = self.terrain.hexGrid
    if origin is None:
        origin = grid.middle
    origin_pos = grid.index_to_hexposition(origin)

    # Step 1: Lazy init coarse basin (O(n²) only once)
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()

    # Step 2: Base terrain (cached)
    result = self.db.load_or_generate_chunk(
        self, origin=origin, scale=scale, force_regenerate=force_regenerate
    )
    zoomed = result.data

    # Step 3: Weather
    if compute_weather:
        if not force_regenerate and self.db.has_zoomed_weather(self.ident, origin_pos, scale):
            self.db.load_zoomed_weather(self.ident, origin_pos, scale, zoomed)
        else:
            zoomed.climate = self.terrain.climate
            zoomed.geo = self.terrain.geo
            zoomed.compute_weather()
            self.db.save_zoomed_weather(self.ident, origin_pos, scale, zoomed)

    # Step 4: Build fine_regions mapping (O(n), cheap)
    fine_regions = self._build_fine_regions_map(origin, scale, zoomed.hexGrid)

    # Step 5: Project watersheds (O(n), uses cached coarse basin)
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, zoomed)

    # Step 6: Return DrainageBasins with zoomed terrain
    zoomed_basins = DrainageBasins(zoomed, compute=False)
    zoomed_basins.sheds = zoomed_sheds

    return zoomed_basins

In [ ]:
#| export
@patch
def invalidate_all_caches(self: GeoStorage, cover_id: int) -> dict:
    """Invalidate all cached data when coarse terrain changes."""
    results = {}
    try:
        with self.db.conn:
            results['terrain'] = self.invalidate_chunk_cache(cover_id)

            cursor = self.db.execute("""
                DELETE FROM hex_weather
                WHERE world_id = ? AND scale_level > 0
            """, [cover_id])
            results['weather'] = SaveResult(cursor.rowcount, 'invalidated',
                                           f'{cursor.rowcount} weather records')

            cursor = self.db.execute("""
                DELETE FROM hex_data
                WHERE world_id = ? AND scale_level > 0 AND watershed_id IS NOT NULL
            """, [cover_id])
            results['watersheds'] = SaveResult(cursor.rowcount, 'invalidated',
                                              f'{cursor.rowcount} watershed assignments')

            return results
    except Exception as e:
        return {'error': SaveResult(None, 'error', str(e))}

## Math note: coarse-to-fine index mapping

When we zoom by `scale`, each coarse hex becomes a `scale × scale` block
of fine hexes. This function builds a dict mapping each coarse hex index
to the set of fine hex indices it "owns".

Think of it like a low-res image being upscaled: pixel (3,4) at 2× zoom
becomes fine pixels {(6,8), (6,9), (7,8), (7,9)}.

In [ ]:
#| export
@patch
def _build_fine_regions_map(self: ChunkCover,
                            origin: int,
                            scale: int,
                            fine_grid: HexGrid) -> dict[int, set[int]]:
    """Map coarse hex indices to sets of fine hex indices.

    O(n) — just coordinate math, no flow computation.
    """
    coarse_grid = self.terrain.hexGrid
    extent = 3 * self.rings + self.halo_rings

    center_row = origin // coarse_grid.nCols
    center_col = origin % coarse_grid.nCols

    fine_regions = {}
    invalid = getattr(fine_grid, 'invalidRegion', set())

    for coarse_row in range(center_row - extent, center_row + extent + 1):
        for coarse_col in range(center_col - extent, center_col + extent + 1):
            if not (0 <= coarse_row < coarse_grid.nRows and 0 <= coarse_col < coarse_grid.nCols):
                continue

            coarse_idx = coarse_row * coarse_grid.nCols + coarse_col

            fine_start_row = (coarse_row - (center_row - extent)) * scale
            fine_start_col = (coarse_col - (center_col - extent)) * scale

            fine_hexes = set()
            for dr in range(scale):
                for dc in range(scale):
                    fine_row = fine_start_row + dr
                    fine_col = fine_start_col + dc
                    if 0 <= fine_row < fine_grid.nRows and 0 <= fine_col < fine_grid.nCols:
                        fine_idx = fine_row * fine_grid.nCols + fine_col
                        if fine_idx not in invalid:
                            fine_hexes.add(fine_idx)

            if fine_hexes:
                fine_regions[coarse_idx] = fine_hexes

    return fine_regions

##  Watershed Persistence — save / load / has

In [ ]:
#| export
@patch
def save_chunk_watersheds(self: GeoStorage, chunk: HexChunk,
                          chunk_ref: ChunkRef, world_id: int) -> SaveResult:
    """Save watershed assignments for a chunk."""
    now = int(datetime.now().timestamp())
    p = chunk_ref.position

    if 'watershed_id' not in chunk.fields:
        return SaveResult(None, 'error', 'Chunk missing watershed_id field')

    try:
        with self.db.conn:
            count = 0
            for idx in chunk.iter_core():
                world_pos = chunk.index_to_world(idx)
                ws_id = int(chunk.fields['watershed_id'][idx])
                if ws_id < 0:
                    continue

                self.hexes.insert({
                    'world_id': world_id,
                    'q': world_pos.q, 'r': world_pos.r, 's': world_pos.s,
                    'grid_index': idx,
                    'elevation': float(chunk.elevations[idx]),
                    'watershed_id': ws_id,
                    'modified': now
                })
                count += 1

            return SaveResult(count, 'saved',
                            f'{count} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_chunk_watersheds(self: GeoStorage, world_id: int,
                          chunk: HexChunk, chunk_ref: ChunkRef) -> LoadResult:
    """Load watershed assignments for a chunk."""
    try:
        p = chunk_ref.position
        hex_rows = []
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            rows = self.query_hexes_in_radius(world_id, world_pos.q, world_pos.r, world_pos.s, radius=0)
            if rows:
                hex_rows.append((idx, rows[0]))

        if not hex_rows:
            return LoadResult(chunk, 'not_found',
                            f'No watershed data for chunk ({p.q},{p.r},{p.s})')

        if 'watershed_id' not in chunk.fields:
            chunk.add_field('watershed_id', default=-1)

        filled = 0
        for idx, row in hex_rows:
            ws_id = row.get('watershed_id')
            if ws_id is not None and ws_id >= 0:
                chunk.fields['watershed_id'][idx] = ws_id
                filled += 1

        return LoadResult(chunk, 'loaded',
                         f'{filled} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return LoadResult(None, 'error', str(e))


@patch
def has_chunk_watersheds(self: GeoStorage, world_id: int,
                         chunk_ref: ChunkRef,
                         min_count: int = 1) -> bool:
    """Check if watershed data exists for chunk."""
    try:
        center = chunk_ref.center_hex
        radius = chunk_ref.rings
        rows = self.query_hexes_in_radius(world_id, center.q, center.r, center.s, radius=radius)
        ws_count = sum(1 for r in rows if r.get('watershed_id') is not None and r.get('watershed_id') >= 0)
        return ws_count >= min_count
    except Exception:
        return False

## GeoStorageDebugger

In [ ]:
#| export
class GeoStorageDebugger:
    """Test harness for GeoStorage with automatic cleanup."""

    def __init__(self, keep_on_error=False):
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = GeoStorage(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False

    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")

    def close(self):
        """Clean up test database."""
        self.server.db.close()
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()

##  DataProxy — Lightweight Chunk Holder

### Math note: why DataProxy exists

When stitching multiple chunks together, the old approach decoded each
chunk into a full `Terrain` with `HexGrid` geometry (O(n) hex objects
per chunk). For 20 chunks that's 400K hex objects created then thrown
away.

`DataProxy` holds only raw numpy arrays (elevations, watersheds) plus
the grid dimensions. We stitch these flat arrays with numpy 2D slicing,
then build the HexGrid geometry **once** at the end on the merged result.

This cuts stitching from ~25s to ~3-4s.


In [ ]:
#| export
@dataclass
class DataProxy:
    """Lightweight chunk data holder for efficient stitching.

    Holds only raw numpy arrays without HexGrid geometry.
    """
    nRows: int
    nCols: int
    radius: float
    elevations: np.ndarray          # Shape: (nRows * nCols,)
    watersheds: Optional[np.ndarray] = None  # Shape: (nRows * nCols,)
    chunk_position: Optional[HexPosition] = None
    invalidRegion: set[int] = field(default_factory=set)
    expected_count: int = 0
    actual_count: int = 0

    @property
    def size(self) -> int:
        return self.nRows * self.nCols

    def __len__(self):
        return self.size

    @property
    def is_complete(self) -> bool:
        """Check if all expected hexes were loaded."""
        if self.expected_count == 0:
            return True
        return self.actual_count >= self.expected_count * 0.95

    def to_dict(self) -> dict:
        """Convert to dict for debugging."""
        return {
            'nRows': self.nRows, 'nCols': self.nCols,
            'radius': self.radius, 'size': self.size,
            'has_watersheds': self.watersheds is not None,
            'chunk_position': str(self.chunk_position) if self.chunk_position else None,
            'invalid_count': len(self.invalidRegion),
            'complete': self.is_complete,
            'expected': self.expected_count, 'actual': self.actual_count
        }

In [ ]:
#| export
@patch
def _load_chunk_as_proxy(self: GeoStorage,
                         cover_id: int,
                         origin_pos: HexPosition,
                         scale: int) -> LoadResult:
    """Load cached chunk as DataProxy (fully numpy-optimized)."""
    try:
        meta = list(self.chunk_meta.rows_where(
            "world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ? AND scale_level = ?",
            [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale]
        ))

        if not meta:
            return LoadResult(None, 'not_found', 'No chunk metadata')

        meta = meta[0]
        nRows = meta['nRows']
        nCols = meta['nCols']
        radius = meta['radius']
        expected_count = meta['hex_count']

        raw_rows = self.db.execute("""
            SELECT grid_index, elevation, watershed_id
            FROM hex_data
            WHERE world_id = ?
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale]).fetchall()

        if not raw_rows:
            return LoadResult(None, 'not_found', 'No cached hex data')

        now = int(datetime.now().timestamp())
        self.chunk_meta.update({
            'id': meta['id'],
            'last_accessed': now,
            'access_count': meta['access_count'] + 1
        })

        grid_size = nRows * nCols
        elevations = np.full(grid_size, -100.0)
        watersheds = np.full(grid_size, -1, dtype=int)

        indices = np.array([row[0] for row in raw_rows], dtype=int)
        elev_values = np.array([row[1] for row in raw_rows])
        ws_values = np.array([row[2] if row[2] is not None else -1
                              for row in raw_rows], dtype=int)

        valid_mask = (indices >= 0) & (indices < grid_size)
        valid_indices = indices[valid_mask]

        elevations[valid_indices] = elev_values[valid_mask]
        watersheds[valid_indices] = ws_values[valid_mask]

        water_mask = elev_values[valid_mask] <= -99.0
        invalidRegion = set(valid_indices[water_mask].tolist())

        actual_count = int(np.sum(elevations > -99.0))

        proxy = DataProxy(
            nRows=nRows, nCols=nCols, radius=radius,
            elevations=elevations, watersheds=watersheds,
            chunk_position=origin_pos, invalidRegion=invalidRegion,
            expected_count=expected_count, actual_count=actual_count
        )

        if not proxy.is_complete:
            return LoadResult(proxy, 'partial',
                            f'{actual_count}/{expected_count} hexes (incomplete)')

        return LoadResult(proxy, 'loaded', f'{len(raw_rows)} hexes as DataProxy')

    except Exception as e:
        return LoadResult(None, 'error', str(e))

## stitch_chunks_fast

### Math note: 2D numpy block copy

Each chunk is a flat array of length `nRows × nCols`. To stitch
chunks side-by-side we:

1. Reshape each chunk to 2D: `(nRows, nCols)`
2. Compute its offset in the merged grid based on its coarse grid position
3. Copy the 2D block into the merged array with a single slice assignment

This is **much** faster than looping over individual hexes because
numpy's slice assignment runs in C.

In [ ]:
#| export
def stitch_chunks_fast(chunks: set[ChunkRef],
                       cover: ChunkCover,
                       storage: GeoStorage,
                       scale: int) -> tuple[DataProxy, dict[tuple, tuple], int, int]:
    """Load chunks as DataProxy and stitch using numpy block copies.

    Generates chunks on cache miss, then loads as lightweight proxy.
    Returns (stitched_proxy, chunk_offsets, chunk_rows, chunk_cols).
    """
    if not chunks:
        return None, {}, 0, 0

    coarse_grid = cover.terrain.hexGrid

    chunk_proxies = {}
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue

        origin_pos = coarse_grid.index_to_hexposition(origin_idx)

        result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)

        if result.status == 'not_found':
            gen = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
            if gen.status in ('generated', 'cached'):
                result = storage._load_chunk_as_proxy(cover.ident, origin_pos, scale)
            else:
                continue

        if result.status == 'error':
            continue

        if result.status in ('loaded', 'partial'):
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_proxies[chunk.key] = (result.data, coarse_row, coarse_col)

    if not chunk_proxies:
        return None, {}, 0, 0

    first_proxy = next(iter(chunk_proxies.values()))[0]
    chunk_rows = first_proxy.nRows
    chunk_cols = first_proxy.nCols
    radius = first_proxy.radius

    mismatched = {k for k, (p, _, _) in chunk_proxies.items()
                  if p.nRows != chunk_rows or p.nCols != chunk_cols}
    if mismatched:
        for k in mismatched:
            del chunk_proxies[k]

    if not chunk_proxies:
        return None, {}, 0, 0

    coarse_rows = [d[1] for d in chunk_proxies.values()]
    coarse_cols = [d[2] for d in chunk_proxies.values()]
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)

    merged_rows = (max_coarse_row - min_coarse_row) * scale + chunk_rows
    merged_cols = (max_coarse_col - min_coarse_col) * scale + chunk_cols

    merged_elev = np.full((merged_rows, merged_cols), -100.0)
    merged_ws = np.full((merged_rows, merged_cols), -1, dtype=int)
    merged_invalid = set()

    chunk_offsets = {}

    for key, (proxy, coarse_row, coarse_col) in chunk_proxies.items():
        row_off = (coarse_row - min_coarse_row) * scale
        col_off = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_off, col_off)

        p_elev = proxy.elevations.reshape(proxy.nRows, proxy.nCols)
        p_ws = proxy.watersheds.reshape(proxy.nRows, proxy.nCols)

        sr0, sr1 = 0, proxy.nRows
        sc0, sc1 = 0, proxy.nCols
        dr0, dr1 = row_off, row_off + proxy.nRows
        dc0, dc1 = col_off, col_off + proxy.nCols

        if dr0 < 0:           sr0 -= dr0; dr0 = 0
        if dr1 > merged_rows: sr1 -= (dr1 - merged_rows); dr1 = merged_rows
        if dc0 < 0:           sc0 -= dc0; dc0 = 0
        if dc1 > merged_cols: sc1 -= (dc1 - merged_cols); dc1 = merged_cols

        if dr0 < dr1 and dc0 < dc1:
            merged_elev[dr0:dr1, dc0:dc1] = p_elev[sr0:sr1, sc0:sc1]
            merged_ws[dr0:dr1, dc0:dc1] = p_ws[sr0:sr1, sc0:sc1]

        for local_idx in proxy.invalidRegion:
            lr = local_idx // proxy.nCols
            lc = local_idx % proxy.nCols
            mr = row_off + lr
            mc = col_off + lc
            if 0 <= mr < merged_rows and 0 <= mc < merged_cols:
                merged_invalid.add(mr * merged_cols + mc)

    merged_proxy = DataProxy(
        nRows=merged_rows, nCols=merged_cols, radius=radius,
        elevations=merged_elev.ravel(),
        watersheds=merged_ws.ravel(),
        invalidRegion=merged_invalid,
        expected_count=sum(p[0].expected_count for p in chunk_proxies.values()),
        actual_count=sum(p[0].actual_count for p in chunk_proxies.values())
    )

    return merged_proxy, chunk_offsets, chunk_rows, chunk_cols

In [ ]:
#| export
def proxy_to_terrain(proxy: DataProxy,
                     climate: ClimatePreset = None,
                     geo: GeoBounds = None) -> Terrain:
    """Convert DataProxy to full Terrain with geometry.

    Builds HexGrid once on the final merged result.
    """
    bounds = MapRect(
        MapCord(0, 0),
        MapSize(proxy.nRows * proxy.radius, proxy.nCols * proxy.radius)
    )

    terrain = Terrain(
        bounds=bounds,
        radius=proxy.radius,
        climate=climate,
        geo=geo
    )

    terrain.hexGrid.nRows = proxy.nRows
    terrain.hexGrid.nCols = proxy.nCols
    terrain.hexGrid.adjustRadius(proxy.radius)

    terrain.elevations = proxy.elevations.copy()

    if proxy.watersheds is not None and np.any(proxy.watersheds >= 0):
        terrain.fields['watershed_id'] = proxy.watersheds.copy()

    terrain.hexGrid.invalidRegion = proxy.invalidRegion.copy()

    return terrain

##  Region Zoom Helpers — bounding box and chunk discovery

In [ ]:
#| export
@dataclass
class ZoomResult:
    """Result of zooming into a region."""
    terrain: Terrain
    basins: DrainageBasins
    mapper: Callable[[int], int]  # fine_idx → coarse_idx (-1 if invalid)
    chunks_loaded: int


def region_bounding_box(region: HexRegion) -> tuple[int, int, int, int]:
    """Get (min_row, max_row, min_col, max_col) for a region."""
    grid = region.hexGrid
    rows = []
    cols = []
    for idx in region.hexes:
        row, col = grid.index_to_row_col(idx)
        rows.append(row)
        cols.append(col)
    return min(rows), max(rows), min(cols), max(cols)


def bbox_to_chunk_refs(min_row: int, max_row: int, min_col: int, max_col: int,
                       cover: ChunkCover) -> set[ChunkRef]:
    """Find all ChunkRefs that overlap a bounding box."""
    grid = cover.terrain.hexGrid
    chunk_rings = cover.rings
    spacing = chunk_rings * 2

    chunks = set()

    for row in range(min_row, max_row + 1, spacing):
        for col in range(min_col, max_col + 1, spacing):
            idx = grid.row_col_to_index(row, col)
            if idx < 0:
                continue
            world_pos = grid.index_to_hexposition(idx, origin_index=0)
            chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
            chunks.add(ChunkRef(chunk_pos, chunk_rings))

    for row in [min_row, max_row]:
        for col in [min_col, max_col]:
            idx = grid.row_col_to_index(row, col)
            if idx >= 0:
                world_pos = grid.index_to_hexposition(idx, origin_index=0)
                chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
                chunks.add(ChunkRef(chunk_pos, chunk_rings))

    return chunks

## build_fine_to_coarse_mapper

### Math note: the mapper function

The mapper converts a fine-grid hex index back to the coarse-grid hex
it came from. This is the inverse of the zoom operation.

Given a fine hex at `(fine_row, fine_col)`:

1. **Which chunk?** — find which chunk's offset range contains this hex
2. **Local position** — subtract the chunk's offset to get local coords
3. **Scale down** — integer-divide by `scale` to get coarse-local coords
4. **To coarse grid** — add the chunk center's coarse grid position

This works on DataProxy (uses `fine_nCols` instead of requiring a
full Terrain object).

In [ ]:
#| export
def build_fine_to_coarse_mapper(cover: ChunkCover,
                                fine_nCols: int,
                                chunk_offsets: dict,
                                scale: int,
                                chunk_rows: int,
                                chunk_cols: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index.

    Only needs fine_nCols (not a full Terrain) so it works on DataProxy
    before HexGrid construction.
    """
    coarse_grid = cover.terrain.hexGrid
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}

    def mapper(fine_idx: int) -> int:
        fine_row = fine_idx // fine_nCols
        fine_col = fine_idx % fine_nCols

        best_key = None
        local_row = local_col = 0
        for (row_off, col_off), key in offset_to_chunk.items():
            if row_off <= fine_row < row_off + chunk_rows and \
               col_off <= fine_col < col_off + chunk_cols:
                best_key = key
                local_row = fine_row - row_off
                local_col = fine_col - col_off
                break

        if best_key is None:
            return -1

        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale

        q, r, s = best_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1

        center_row, center_col = coarse_grid.index_to_row_col(center_idx)

        half_rows = chunk_rows // (2 * scale)
        half_cols = chunk_cols // (2 * scale)

        coarse_row = center_row - half_rows + coarse_local_row
        coarse_col = center_col - half_cols + coarse_local_col

        return coarse_grid.row_col_to_index(coarse_row, coarse_col)

    return mapper

Can you go into more details about the build_fine_to_coarse_mapper. Give some equationa and examples? This is going to be a key object so we want future ai requests to completely understand it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

## `build_fine_to_coarse_mapper` — Deep Dive

This function answers the question: **"Given a hex index in the stitched fine (zoomed) grid, which hex in the original coarse grid did it come from?"**

It returns a closure `mapper(fine_idx) → coarse_idx` (or `-1` if unmappable).

### The coordinate chain

There are **four** coordinate spaces involved:

| Space | Description | Example |
|---|---|---|
| **Fine flat index** | Single int in the stitched merged grid | `fine_idx = 45` |
| **Fine (row, col)** | 2D position in the merged grid | `(3, 3)` |
| **Chunk-local (row, col)** | Position within one chunk's fine grid | `(3, 3)` if offset is `(0,0)` |
| **Coarse (row, col)** | Position in the original coarse grid | `(3, 6)` |

The mapper walks backwards through these:

```
fine_idx → fine (row,col) → chunk-local (row,col) → coarse (row,col) → coarse_idx
```

### Step-by-step equations

**Step 1 — Unflatten the fine index:**

$$\text{fine\_row} = \lfloor \text{fine\_idx} \;/\; \text{fine\_nCols} \rfloor$$
$$\text{fine\_col} = \text{fine\_idx} \mod \text{fine\_nCols}$$

**Step 2 — Identify which chunk owns this hex:**

Each chunk was placed at offset $(r_{\text{off}}, c_{\text{off}})$ in the merged grid during stitching. A fine hex belongs to chunk $k$ if:

$$r_{\text{off}}^{(k)} \leq \text{fine\_row} < r_{\text{off}}^{(k)} + H_{\text{chunk}}$$
$$c_{\text{off}}^{(k)} \leq \text{fine\_col} < c_{\text{off}}^{(k)} + W_{\text{chunk}}$$

where $H_{\text{chunk}}$ and $W_{\text{chunk}}$ are `chunk_rows` and `chunk_cols` — the dimensions of each zoomed chunk (all chunks are the same size).

**Step 3 — Compute chunk-local position:**

$$\text{local\_row} = \text{fine\_row} - r_{\text{off}}^{(k)}$$
$$\text{local\_col} = \text{fine\_col} - c_{\text{off}}^{(k)}$$

**Step 4 — Scale down (the key step):**

Integer division by `scale` reverses the zoom. Each `scale × scale` block of fine hexes maps to one coarse hex:

$$\text{coarse\_local\_row} = \lfloor \text{local\_row} \;/\; \text{scale} \rfloor$$
$$\text{coarse\_local\_col} = \lfloor \text{local\_col} \;/\; \text{scale} \rfloor$$

This means fine hexes `(0,0), (0,1), (1,0), (1,1)` all map to coarse-local `(0,0)` when `scale=2`.

**Step 5 — Anchor to coarse grid via chunk center:**

The chunk key $(q_k, r_k, s_k)$ gives the chunk's position in chunk-space. Its center in the coarse grid is:

$$\text{chunk\_center} = (q_k, r_k, s_k) \times 2 \cdot \text{rings}$$

Convert that to coarse `(center_row, center_col)`, then:

$$h_r = \lfloor H_{\text{chunk}} \;/\; (2 \cdot \text{scale}) \rfloor \qquad h_c = \lfloor W_{\text{chunk}} \;/\; (2 \cdot \text{scale}) \rfloor$$

These are the half-extents: how many coarse hexes the chunk spans in each direction from its center.

$$\text{coarse\_row} = \text{center\_row} - h_r + \text{coarse\_local\_row}$$
$$\text{coarse\_col} = \text{center\_col} - h_c + \text{coarse\_local\_col}$$

**Step 6 — Flatten back:**

$$\text{coarse\_idx} = \text{coarse\_row} \times \text{coarse\_nCols} + \text{coarse\_col}$$

---

### Worked example

Setup:
- `rings = 3`, so `spacing = 6`
- `scale = 2`
- One chunk at key `(1, 0, -1)`
- Zoomed chunk dimensions: `chunk_rows = 14`, `chunk_cols = 14`
- This chunk's stitch offset: `(row_off=0, col_off=0)`
- Coarse grid center for this chunk is at `(center_row=5, center_col=8)`
- Merged grid has `fine_nCols = 14`

Query: **Where does `fine_idx = 45` come from?**

```
Step 1:  fine_row = 45 // 14 = 3      fine_col = 45 % 14 = 3

Step 2:  Chunk (1,0,-1) owns it since 0 ≤ 3 < 14 in both dims

Step 3:  local_row = 3 - 0 = 3        local_col = 3 - 0 = 3

Step 4:  coarse_local_row = 3 // 2 = 1
         coarse_local_col = 3 // 2 = 1

Step 5:  h_r = 14 // (2×2) = 3        h_c = 14 // (2×2) = 3
         coarse_row = 5 - 3 + 1 = 3
         coarse_col = 8 - 3 + 1 = 6

Step 6:  coarse_idx = 3 × coarse_nCols + 6
```

Note that `fine_idx = 44` (which is fine position `(3, 2)`) also maps to the same coarse hex — `coarse_local = (3//2, 2//2) = (1, 1)` — because they're in the same `2×2` block.

### Visual: the scale=2 block mapping

```
Fine grid (one chunk, 6×6):       Coarse grid (3×3):

 ┌──┬──┬──┬──┬──┬──┐              ┌──┬──┬──┐
 │ A│ A│ B│ B│ C│ C│              │ A│ B│ C│
 │ A│ A│ B│ B│ C│ C│              ├──┼──┼──┤
 ├──┼──┼──┼──┼──┼──┤              │ D│ E│ F│
 │ D│ D│ E│ E│ F│ F│              ├──┼──┼──┤
 │ D│ D│ E│ E│ F│ F│              │ G│ H│ I│
 ├──┼──┼──┼──┼──┼──┤              └──┴──┴──┘
 │ G│ G│ H│ H│ I│ I│
 │ G│ G│ H│ H│ I│ I│
 └──┴──┴──┴──┴──┴──┘

 Each 2×2 block of fine hexes → 1 coarse hex
```

### Multi-chunk scenario

When multiple chunks are stitched, the offsets shift:

```
Chunk (0,0,0) at offset (0, 0)     Chunk (1,0,-1) at offset (0, 14)
┌──────────────┬──────────────┐
│  fine_col 0  │  fine_col 14 │
│   to 13      │   to 27      │
└──────────────┴──────────────┘
```

A fine hex at `fine_col = 17` falls in chunk `(1,0,-1)` with `local_col = 17 - 14 = 3`. The rest proceeds as before, but using chunk `(1,0,-1)`'s center for the coarse grid anchor.

### Edge cases

- **Returns `-1`** if the fine hex doesn't fall inside any chunk's offset range (can happen at stitching boundaries)
- **Returns `-1`** if `coarse_grid.hexposition_to_index` can't resolve the chunk center (chunk is outside the coarse grid extent)
- The `offset_to_chunk` reverse-lookup dict means the first matching chunk wins — chunks shouldn't overlap in a correctly stitched grid

### Why it takes `fine_nCols` instead of a Terrain

The mapper is built **before** `proxy_to_terrain()` runs, so there's no `HexGrid` yet — just a `DataProxy` with known dimensions. All we need is `fine_nCols` to unflatten the index. This keeps the entire stitch pipeline in numpy-land until the very end.

## `build_fine_to_coarse_mapper` — Deep Dive

This function answers the question: **"Given a hex index in the stitched fine (zoomed) grid, which hex in the original coarse grid did it come from?"**

It returns a closure `mapper(fine_idx) → coarse_idx` (or `-1` if unmappable).

### The coordinate chain

There are **four** coordinate spaces involved:

| Space | Description | Example |
|---|---|---|
| **Fine flat index** | Single int in the stitched merged grid | `fine_idx = 45` |
| **Fine (row, col)** | 2D position in the merged grid | `(3, 3)` |
| **Chunk-local (row, col)** | Position within one chunk's fine grid | `(3, 3)` if offset is `(0,0)` |
| **Coarse (row, col)** | Position in the original coarse grid | `(3, 6)` |

The mapper walks backwards through these:

```
fine_idx → fine (row,col) → chunk-local (row,col) → coarse (row,col) → coarse_idx
```

### Step-by-step equations

**Step 1 — Unflatten the fine index:**

$$\text{fine\_row} = \lfloor \text{fine\_idx} \;/\; \text{fine\_nCols} \rfloor$$
$$\text{fine\_col} = \text{fine\_idx} \mod \text{fine\_nCols}$$

**Step 2 — Identify which chunk owns this hex:**

Each chunk was placed at offset $(r_{\text{off}}, c_{\text{off}})$ in the merged grid during stitching. A fine hex belongs to chunk $k$ if:

$$r_{\text{off}}^{(k)} \leq \text{fine\_row} < r_{\text{off}}^{(k)} + H_{\text{chunk}}$$
$$c_{\text{off}}^{(k)} \leq \text{fine\_col} < c_{\text{off}}^{(k)} + W_{\text{chunk}}$$

where $H_{\text{chunk}}$ and $W_{\text{chunk}}$ are `chunk_rows` and `chunk_cols` — the dimensions of each zoomed chunk (all chunks are the same size).

**Step 3 — Compute chunk-local position:**

$$\text{local\_row} = \text{fine\_row} - r_{\text{off}}^{(k)}$$
$$\text{local\_col} = \text{fine\_col} - c_{\text{off}}^{(k)}$$

**Step 4 — Scale down (the key step):**

Integer division by `scale` reverses the zoom. Each `scale × scale` block of fine hexes maps to one coarse hex:

$$\text{coarse\_local\_row} = \lfloor \text{local\_row} \;/\; \text{scale} \rfloor$$
$$\text{coarse\_local\_col} = \lfloor \text{local\_col} \;/\; \text{scale} \rfloor$$

This means fine hexes `(0,0), (0,1), (1,0), (1,1)` all map to coarse-local `(0,0)` when `scale=2`.

**Step 5 — Anchor to coarse grid via chunk center:**

The chunk key $(q_k, r_k, s_k)$ gives the chunk's position in chunk-space. Its center in the coarse grid is:

$$\text{chunk\_center} = (q_k, r_k, s_k) \times 2 \cdot \text{rings}$$

Convert that to coarse `(center_row, center_col)`, then:

$$h_r = \lfloor H_{\text{chunk}} \;/\; (2 \cdot \text{scale}) \rfloor \qquad h_c = \lfloor W_{\text{chunk}} \;/\; (2 \cdot \text{scale}) \rfloor$$

These are the half-extents: how many coarse hexes the chunk spans in each direction from its center.

$$\text{coarse\_row} = \text{center\_row} - h_r + \text{coarse\_local\_row}$$
$$\text{coarse\_col} = \text{center\_col} - h_c + \text{coarse\_local\_col}$$

**Step 6 — Flatten back:**

$$\text{coarse\_idx} = \text{coarse\_row} \times \text{coarse\_nCols} + \text{coarse\_col}$$

---

### Worked example

Setup:
- `rings = 3`, so `spacing = 6`
- `scale = 2`
- One chunk at key `(1, 0, -1)`
- Zoomed chunk dimensions: `chunk_rows = 14`, `chunk_cols = 14`
- This chunk's stitch offset: `(row_off=0, col_off=0)`
- Coarse grid center for this chunk is at `(center_row=5, center_col=8)`
- Merged grid has `fine_nCols = 14`

Query: **Where does `fine_idx = 45` come from?**

```
Step 1:  fine_row = 45 // 14 = 3      fine_col = 45 % 14 = 3

Step 2:  Chunk (1,0,-1) owns it since 0 ≤ 3 < 14 in both dims

Step 3:  local_row = 3 - 0 = 3        local_col = 3 - 0 = 3

Step 4:  coarse_local_row = 3 // 2 = 1
         coarse_local_col = 3 // 2 = 1

Step 5:  h_r = 14 // (2×2) = 3        h_c = 14 // (2×2) = 3
         coarse_row = 5 - 3 + 1 = 3
         coarse_col = 8 - 3 + 1 = 6

Step 6:  coarse_idx = 3 × coarse_nCols + 6
```

Note that `fine_idx = 44` (which is fine position `(3, 2)`) also maps to the same coarse hex — `coarse_local = (3//2, 2//2) = (1, 1)` — because they're in the same `2×2` block.

### Visual: the scale=2 block mapping

```
Fine grid (one chunk, 6×6):       Coarse grid (3×3):

 ┌──┬──┬──┬──┬──┬──┐              ┌──┬──┬──┐
 │ A│ A│ B│ B│ C│ C│              │ A│ B│ C│
 │ A│ A│ B│ B│ C│ C│              ├──┼──┼──┤
 ├──┼──┼──┼──┼──┼──┤              │ D│ E│ F│
 │ D│ D│ E│ E│ F│ F│              ├──┼──┼──┤
 │ D│ D│ E│ E│ F│ F│              │ G│ H│ I│
 ├──┼──┼──┼──┼──┼──┤              └──┴──┴──┘
 │ G│ G│ H│ H│ I│ I│
 │ G│ G│ H│ H│ I│ I│
 └──┴──┴──┴──┴──┴──┘

 Each 2×2 block of fine hexes → 1 coarse hex
```

### Multi-chunk scenario

When multiple chunks are stitched, the offsets shift:

```
Chunk (0,0,0) at offset (0, 0)     Chunk (1,0,-1) at offset (0, 14)
┌──────────────┬──────────────┐
│  fine_col 0  │  fine_col 14 │
│   to 13      │   to 27      │
└──────────────┴──────────────┘
```

A fine hex at `fine_col = 17` falls in chunk `(1,0,-1)` with `local_col = 17 - 14 = 3`. The rest proceeds as before, but using chunk `(1,0,-1)`'s center for the coarse grid anchor.

### Edge cases

- **Returns `-1`** if the fine hex doesn't fall inside any chunk's offset range (can happen at stitching boundaries)
- **Returns `-1`** if `coarse_grid.hexposition_to_index` can't resolve the chunk center (chunk is outside the coarse grid extent)
- The `offset_to_chunk` reverse-lookup dict means the first matching chunk wins — chunks shouldn't overlap in a correctly stitched grid

### Why it takes `fine_nCols` instead of a Terrain

The mapper is built **before** `proxy_to_terrain()` runs, so there's no `HexGrid` yet — just a `DataProxy` with known dimensions. All we need is `fine_nCols` to unflatten the index. This keeps the entire stitch pipeline in numpy-land until the very end.

Do I need to put the $$ in a tag to show the math in the above note

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

In solveit, notes are rendered as standard markdown. Whether `$$` math blocks render as formatted equations depends on whether the instance has a math rendering library (like MathJax or KaTeX) loaded.

The quickest way to find out: just look at your note — if you see the formatted equations, it's working. If you see the raw `$$\text{fine\_row} = ...$$` text, then math rendering isn't active.

If it's **not** rendering, you have a few options:

1. **Drop the LaTeX and use code blocks** — since this is reference documentation for AI context (not a textbook), plain-text math in fenced blocks is arguably *better* for future AI consumption anyway:

```
fine_row = fine_idx // fine_nCols
fine_col = fine_idx % fine_nCols
```

2. **Use inline code** for formulas — e.g. `` `coarse_local_row = local_row // scale` `` — which renders everywhere and is unambiguous to both humans and AI.

Honestly, since you said the goal is for **future AI requests to completely understand it**, I'd lean toward option 1 regardless. LLMs parse Python-style math and code blocks more reliably than LaTeX notation. The `$$` blocks look nice for humans but add no clarity for an AI reader — and they might even confuse things if the LaTeX gets mangled in the context window.

How could I render it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

A few practical options:

**Option A — Use IPython's Math display from a code cell:**

```python
from IPython.display import Math, display
display(Math(r'\text{fine\_row} = \lfloor \text{fine\_idx} \;/\; \text{fine\_nCols} \rfloor'))
```

This renders inline in the code output using MathJax.

**Option B — Render via matplotlib (produces an image):**

```python
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 0.6))
ax.text(0.5, 0.5, r'$\mathrm{fine\_row} = \lfloor \mathrm{fine\_idx} \;/\; \mathrm{fine\_nCols} \rfloor$',
        fontsize=16, ha='center', va='center')
ax.axis('off')
plt.tight_layout()
plt.show()
```

**Option C — Output HTML with MathJax from a code cell:**

```python
from IPython.display import HTML
HTML("""
<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>
$$\\text{fine\\_row} = \\lfloor \\text{fine\\_idx} \\;/\\; \\text{fine\\_nCols} \\rfloor$$
""")
```

That said — for your stated goal of making this understandable to future AI requests, I'd still recommend the plain-text code-block approach from before. LaTeX renders beautifully for human eyes but adds nothing for AI comprehension, and these workarounds produce *output* (images or rendered HTML) rather than inline note content. The note itself will still show raw `$$` in the AI context regardless of how prettily it displays on screen.

## Proxy Cropping — crop_proxy_to_region and make_cropped_mapper

### Math note: 2D array cropping

After stitching, the merged grid may be much larger than the region
we care about. Rather than building a huge HexGrid, we crop the numpy
array to just the bounding box of the region's fine hexes (plus padding).

This is a pure numpy 2D slice — reshape to `(nRows, nCols)`, slice
`[min_row:max_row+1, min_col:max_col+1]`, flatten back.

The mapper must then be wrapped to translate cropped indices back to
the original merged grid before looking up the coarse hex.

In [ ]:
#| export
def crop_proxy_to_region(proxy: DataProxy,
                         region_fine_indices: set[int],
                         padding: int = 4) -> tuple[DataProxy, int, int, int, int]:
    """Crop DataProxy to bounding box of region's fine indices + padding.

    Returns (cropped_proxy, trim_top, trim_left, old_nRows, old_nCols).
    """
    if not region_fine_indices:
        return proxy, 0, 0, proxy.nRows, proxy.nCols

    old_nRows, old_nCols = proxy.nRows, proxy.nCols

    indices = np.array(list(region_fine_indices))
    rows = indices // old_nCols
    cols = indices % old_nCols

    min_row = max(0, int(rows.min()) - padding)
    max_row = min(old_nRows - 1, int(rows.max()) + padding)
    min_col = max(0, int(cols.min()) - padding)
    max_col = min(old_nCols - 1, int(cols.max()) + padding)

    new_nRows = max_row - min_row + 1
    new_nCols = max_col - min_col + 1

    old_size = old_nRows * old_nCols
    new_size = new_nRows * new_nCols
    if (old_size - new_size) < old_size * 0.1:
        return proxy, 0, 0, old_nRows, old_nCols

    elev_2d = proxy.elevations.reshape(old_nRows, old_nCols)
    cropped_elev = elev_2d[min_row:max_row+1, min_col:max_col+1].ravel().copy()

    cropped_ws = None
    if proxy.watersheds is not None:
        ws_2d = proxy.watersheds.reshape(old_nRows, old_nCols)
        cropped_ws = ws_2d[min_row:max_row+1, min_col:max_col+1].ravel().copy()

    new_invalid = set()
    for old_idx in proxy.invalidRegion:
        old_r = old_idx // old_nCols
        old_c = old_idx % old_nCols
        if min_row <= old_r <= max_row and min_col <= old_c <= max_col:
            new_invalid.add((old_r - min_row) * new_nCols + (old_c - min_col))

    cropped = DataProxy(
        nRows=new_nRows, nCols=new_nCols, radius=proxy.radius,
        elevations=cropped_elev, watersheds=cropped_ws,
        chunk_position=proxy.chunk_position,
        invalidRegion=new_invalid,
        expected_count=proxy.expected_count,
        actual_count=proxy.actual_count
    )

    return cropped, min_row, min_col, old_nRows, old_nCols


def make_cropped_mapper(original_mapper: Callable[[int], int],
                        trim_top: int, trim_left: int,
                        old_nCols: int, new_nCols: int) -> Callable[[int], int]:
    """Wrap a fine→coarse mapper to account for proxy cropping."""
    if trim_top == 0 and trim_left == 0:
        return original_mapper

    def cropped_mapper(cropped_idx: int) -> int:
        row = cropped_idx // new_nCols + trim_top
        col = cropped_idx % new_nCols + trim_left
        return original_mapper(row * old_nCols + col)

    return cropped_mapper

## zoom_region_fast — The Main Pipeline

This is the primary entry point for zooming into a HexRegion.

Pipeline:
1. Compute bounding box of the region (with padding)
2. Find which chunks cover that bounding box
3. Stitch chunks as DataProxy (numpy only, no HexGrid)
4. Build the fine→coarse mapper on the uncropped proxy
5. Identify which fine hexes map to the region; mark invalids
6. Crop the proxy to the region's bounding box
7. Wrap the mapper with the crop offset
8. Build the Terrain **once** from the cropped proxy
9. Project weather fields from coarse → fine using the mapper
10. Project watersheds and return ZoomResult

In [ ]:
#| export
@patch
def zoom_region_fast(self: ChunkCover,
                     region: HexRegion,
                     scale: int = 2,
                     compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region using DataProxy + numpy cropping."""
    if self.db is None:
        raise ValueError("ChunkCover.db must be set")

    # Step 1: Bounding box with padding
    min_row, max_row, min_col, max_col = region_bounding_box(region)

    padding = self.halo_rings + 1
    coarse_grid = self.terrain.hexGrid
    min_row = max(0, min_row - padding)
    max_row = min(coarse_grid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(coarse_grid.nCols - 1, max_col + padding)

    # Step 2: Find and stitch chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)

    merged_proxy, chunk_offsets, chunk_rows, chunk_cols = stitch_chunks_fast(
        chunks, self, self.db, scale
    )

    if merged_proxy is None:
        return ZoomResult(None, None, lambda x: -1, 0)

    # Step 3: Lazy init coarse basin
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()

    # Step 4: Build mapper on UNCROPPED proxy
    uncropped_mapper = build_fine_to_coarse_mapper(
        self, merged_proxy.nCols, chunk_offsets, scale,
        chunk_rows=chunk_rows, chunk_cols=chunk_cols
    )

    # Step 5: Find region's fine indices + mark invalid
    region_hexes = region.hexes
    region_fine_indices = set()

    for fine_idx in range(merged_proxy.size):
        if fine_idx in merged_proxy.invalidRegion:
            continue
        if merged_proxy.elevations[fine_idx] <= -99.0:
            merged_proxy.invalidRegion.add(fine_idx)
            continue

        coarse_idx = uncropped_mapper(fine_idx)
        if coarse_idx < 0:
            merged_proxy.invalidRegion.add(fine_idx)
            continue

        if coarse_idx in region_hexes:
            region_fine_indices.add(fine_idx)

    # Step 6: Crop proxy to region bbox
    crop_padding = max(4, scale * (self.halo_rings + 1))
    cropped_proxy, trim_top, trim_left, old_nRows, old_nCols = crop_proxy_to_region(
        merged_proxy, region_fine_indices, padding=crop_padding
    )

    # Step 7: Wrap mapper with crop offset
    mapper = make_cropped_mapper(
        uncropped_mapper, trim_top, trim_left,
        old_nCols, cropped_proxy.nCols
    )

    # Step 8: Build Terrain ONCE on cropped proxy
    merged_terrain = proxy_to_terrain(
        cropped_proxy,
        climate=getattr(self.terrain, 'climate', None),
        geo=getattr(self.terrain, 'geo', None)
    )

    fine_grid = merged_terrain.hexGrid

    # Step 9: Project weather from coarse → fine
    fine_indices = []
    coarse_indices = []
    if compute_weather:
        coarse_terrain = self.terrain
        weather_fields = ['temperature', 'precipitation', 'humidity',
                         'climate_pet', 'aridity_index', 'climate',
                         'latitude', 'longitude', 'distance_to_coast',
                         'precip_rate_mmh']

        for fine_idx in range(cropped_proxy.size):
            if fine_idx in fine_grid.invalidRegion:
                continue
            coarse_idx = mapper(fine_idx)
            if coarse_idx >= 0:
                fine_indices.append(fine_idx)
                coarse_indices.append(coarse_idx)

        fine_arr = np.array(fine_indices, dtype=int)
        coarse_arr = np.array(coarse_indices, dtype=int)

        for field_name in weather_fields:
            if field_name in coarse_terrain.fields:
                coarse_field = coarse_terrain.fields[field_name]
                valid = coarse_arr < len(coarse_field)
                merged_terrain.fields[field_name] = np.zeros(len(merged_terrain.elevations))
                merged_terrain.fields[field_name][fine_arr[valid]] = coarse_field[coarse_arr[valid]]

        if 'temperature' not in merged_terrain.fields or \
           'precipitation' not in merged_terrain.fields:
            merged_terrain.compute_weather()

    # Step 10: Build fine_regions and project watersheds
    fine_regions = {}
    for fi, ci in zip(fine_indices, coarse_indices):
        if ci not in fine_regions:
            fine_regions[ci] = set()
        fine_regions[ci].add(fi)

    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)

    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds

    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )

## `zoom_region_fast` — Deep Dive

This is the **main pipeline** for the question: *"I have a HexRegion on the coarse grid — give me a high-resolution Terrain covering just that region, with weather and watersheds."*

It orchestrates every piece we've built so far into a 10-step pipeline that stays in numpy-land as long as possible, only constructing the expensive `HexGrid` geometry once at the very end.

### The pipeline at a glance

```
HexRegion (coarse)
    │
    ▼
┌─────────────────┐
│ 1. Bounding box  │  O(|region|)
│ 2. Chunk refs    │  O(bbox area / spacing²)
│ 3. Stitch chunks │  O(chunks × chunk_size)  ← numpy block copies
│ 4. Build mapper  │  O(1) closure creation
│ 5. Region filter │  O(merged_size)  ← identifies fine hexes in region
│ 6. Crop proxy    │  O(cropped_size) ← numpy 2D slice
│ 7. Wrap mapper   │  O(1) closure wrapping
│ 8. Build Terrain │  O(cropped_size) ← HexGrid constructed ONCE here
│ 9. Project weather│  O(valid_fine_hexes) ← vectorized numpy scatter
│10. Project sheds │  O(valid_fine_hexes) ← dict grouping + lookup
└─────────────────┘
    │
    ▼
ZoomResult(terrain, basins, mapper, chunks_loaded)
```

### Step 1 — Bounding box with padding

Given a `HexRegion` (a set of coarse hex indices), we find the axis-aligned bounding box in row/col space:

$$r_{\min}, r_{\max}, c_{\min}, c_{\max} = \text{region\_bounding\_box}(\text{region})$$

Then we **pad** by `halo_rings + 1` in every direction (clamped to grid bounds). The padding ensures that when we zoom, fine hexes at the region boundary have enough neighbor data for interpolation and drainage flow:

$$r_{\min}' = \max(0,\; r_{\min} - p) \qquad r_{\max}' = \min(N_r - 1,\; r_{\max} + p)$$

where $p = \text{halo\_rings} + 1$.

**Why halo_rings?** Each chunk already has a halo of extra hexes for seamless stitching. The `+1` ensures we don't get edge artifacts on the region boundary itself.

### Step 2 — Chunk discovery

`bbox_to_chunk_refs` samples the padded bounding box at `spacing = 2 × rings` intervals and maps each sample point to its chunk via `world_to_chunk`:

```
Padded bbox on coarse grid:
┌─────────────────────────┐
│  ·  ·  ·  ·  ·  ·  ·   │   · = sample points at spacing intervals
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   Each · → world_to_chunk → ChunkRef
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   + corners checked explicitly
└─────────────────────────┘
```

The result is a `set[ChunkRef]` — deduplicated because multiple sample points can land in the same chunk.

**Corner check:** The loop at `spacing` intervals can miss chunks that *just barely* overlap a corner. The explicit corner sampling catches those.

### Step 3 — Stitch chunks (the heavy lift)

`stitch_chunks_fast` does:

1. For each `ChunkRef`, look up the cached zoomed chunk as a `DataProxy`
2. On cache miss, call `load_or_generate_chunk` (runs the actual zoom computation), then reload as `DataProxy`
3. Compute the merged grid dimensions from the range of coarse positions × scale
4. Copy each chunk's 2D elevation/watershed arrays into the merged array at its offset

The key equations for merged dimensions:

$$H_{\text{merged}} = (r_{\max}^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale} + H_{\text{chunk}}$$
$$W_{\text{merged}} = (c_{\max}^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale} + W_{\text{chunk}}$$

And the placement offset for each chunk:

$$\text{row\_off}^{(k)} = (r_k^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale}$$
$$\text{col\_off}^{(k)} = (c_k^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale}$$

This is all numpy 2D slice assignment — no per-hex loops.

### Step 4 — Build the mapper (on uncropped proxy)

This is the `build_fine_to_coarse_mapper` from the previous deep dive. The critical point: it's built on the **uncropped** merged proxy, because we need it in Step 5 before we know what to crop.

### Step 5 — Identify the region's fine hexes

This is the step that connects the coarse `HexRegion` to the fine grid. For every valid fine hex:

$$\text{coarse\_idx} = \text{mapper}(\text{fine\_idx})$$

If `coarse_idx ∈ region.hexes`, this fine hex "belongs" to the region.

```
Coarse region (shaded):          Fine merged grid:
┌──┬──┬──┬──┬──┐                 ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │░░│░░│  │   mapper        │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│░░│  │   ←────────     │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│  │  │                 │  │  │  │  │▓▓│▓▓│  │  │  │  │
│  │  │  │  │  │                 │  │  │  │  │  │  │  │  │  │  │
└──┴──┴──┴──┴──┘                 └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

                                  ▓▓ = fine hexes where mapper → region hex
```

Three types of fine hexes get marked **invalid** here:
- Already in `invalidRegion` (from stitching)
- Elevation ≤ −99.0 (deep water sentinel)
- Mapper returns −1 (outside any chunk)

This is the most expensive loop — `O(merged_size)` — but it's pure Python integer ops, no geometry.

### Step 6 — Crop the proxy

The merged proxy can be much larger than the region. Imagine a region that's 5 coarse hexes in a vertical strip — the merged proxy might be 4 chunks wide but we only need a narrow column.

$$\text{crop\_padding} = \max(4,\; \text{scale} \times (\text{halo\_rings} + 1))$$

The crop finds the bounding box of `region_fine_indices`, pads it, and extracts a numpy 2D slice:

```
Before crop (merged proxy):      After crop:
┌────────────────────────┐        ┌──────────┐
│                        │        │          │
│       ┌──────┐         │   →    │ ┌──────┐ │
│       │region│         │        │ │region│ │
│       └──────┘         │        │ └──────┘ │
│                        │        │          │
└────────────────────────┘        └──────────┘
    50×80 = 4000 cells              20×30 = 600 cells
```

**Bail-out:** If cropping would save less than 10% of cells, we skip it (the overhead isn't worth it).

The function returns the crop offsets `(trim_top, trim_left)` needed to translate cropped coordinates back to merged coordinates.

### Step 7 — Wrap the mapper

Since the proxy was cropped, fine indices shifted. `make_cropped_mapper` wraps the original mapper:

$$\text{merged\_row} = \text{cropped\_row} + \text{trim\_top}$$
$$\text{merged\_col} = \text{cropped\_col} + \text{trim\_left}$$
$$\text{merged\_idx} = \text{merged\_row} \times W_{\text{old}} + \text{merged\_col}$$

Then feeds `merged_idx` to the original uncropped mapper. If no cropping happened (trim = 0), the original mapper is returned as-is.

### Step 8 — Build Terrain ONCE

`proxy_to_terrain(cropped_proxy)` is the **only** place where `HexGrid` geometry gets constructed. This is the payoff of the entire DataProxy design:

- `stitch_chunks_fast` → numpy only
- `crop_proxy_to_region` → numpy only
- Steps 5-7 → integer arithmetic only
- **Here** → create hex objects, compute pixel positions, etc.

We do it on the **cropped** proxy, so the HexGrid is as small as possible.

### Step 9 — Project weather (vectorized)

Weather lives on the coarse terrain. We need it on the fine terrain. The mapper gives us the correspondence, and we vectorize it:

```python
fine_arr   = [  0,   1,   2,   5,   6, ...]   # fine indices (valid)
coarse_arr = [ 42,  42,  42,  43,  43, ...]   # their coarse parents
```

For each weather field (temperature, precipitation, etc.):

$$\text{fine\_field}[\text{fine\_arr}] = \text{coarse\_field}[\text{coarse\_arr}]$$

This is a single numpy **fancy-index scatter** — no Python loop over hexes. Every fine hex in a `scale × scale` block gets the same coarse value (nearest-neighbor projection).

**Fallback:** If temperature or precipitation still isn't populated (e.g. the coarse terrain didn't have them), it calls `merged_terrain.compute_weather()` to generate from scratch.

### Step 10 — Project watersheds

We group the fine→coarse mapping into `fine_regions`:

$$\text{fine\_regions}[c] = \{f \;|\; \text{mapper}(f) = c\}$$

This is exactly the dict that `project_watersheds` expects — it assigns each fine hex the watershed ID of its coarse parent from the `DrainageBasins`.

The coarse `DrainageBasins` is lazily computed once (Step 3's `self.basin` check) and then reused for all subsequent zooms.

---

### Worked example

Setup:
- `cover` with `rings=3`, `halo_rings=1`, coarse grid `20×20`
- Region = 6 coarse hexes forming an L-shape
- `scale = 2`

```
Step 1:  bbox = rows [4,8], cols [6,9]
         padded = rows [2,10], cols [4,11]  (padding = 2)

Step 2:  At spacing=6, samples hit chunks (0,0,0) and (1,0,-1)
         → 2 ChunkRefs

Step 3:  Each chunk zooms to 14×14 fine hexes
         Merged proxy = 14×28 = 392 cells  (chunks side by side)

Step 4:  Mapper closure built (captures chunk_offsets, scale=2, etc.)

Step 5:  Loop over 392 fine cells:
         - 168 map to region hexes → region_fine_indices
         - 180 map to non-region hexes → ignored
         - 44 invalid (water/edge)

Step 6:  region_fine_indices bbox = rows [4,17], cols [4,17]
         Crop to 18×18 = 324 cells (saves 17%)
         trim_top=4, trim_left=4

Step 7:  Mapper wrapped: cropped_idx → +4 to both row,col → original mapper

Step 8:  HexGrid(18, 18, radius=5) built — 324 hexes with geometry

Step 9:  temperature[fine_arr] = coarse_temp[coarse_arr]  (one numpy op per field)

Step 10: fine_regions = {42: {0,1,2,3}, 43: {4,5,6,7}, ...}
         Each set gets its parent's watershed_id
         → ZoomResult with 168 meaningful fine hexes
```

### Performance characteristics

| Step | Cost | Dominates when... |
|------|------|-------------------|
| 3. Stitch | O(chunks × chunk_size) | Many chunks, cache cold |
| 5. Region filter | O(merged_size) | Large merged proxy |
| 8. Build Terrain | O(cropped_size) | Large region |
| 9. Weather | O(valid_fine × fields) | Many weather fields |

The cache-cold case (first zoom) is dominated by Step 3 — actually computing the zoomed terrain. Subsequent zooms hit the cache and Steps 5-9 dominate, which are all numpy-fast.

### The returned ZoomResult

```python
@dataclass
class ZoomResult:
    terrain: Terrain          # Full Terrain with HexGrid, elevations, weather fields
    basins: DrainageBasins    # Watersheds projected from coarse → fine
    mapper: Callable          # cropped_fine_idx → coarse_idx (for further lookups)
    chunks_loaded: int        # How many chunks were involved
```

The `mapper` is returned so callers can do their own fine→coarse lookups (e.g., "which biome does this fine hex inherit?" or "what political territory is it in?"). It's the cropped version, so it works directly with indices into `terrain.hexGrid`.

I really loved the full explanation for build_fine_to_coarse_mapper can you do something for zoom_region_fast

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

## `zoom_region_fast` — Deep Dive

This is the **main pipeline** for the question: *"I have a HexRegion on the coarse grid — give me a high-resolution Terrain covering just that region, with weather and watersheds."*

It orchestrates every piece we've built so far into a 10-step pipeline that stays in numpy-land as long as possible, only constructing the expensive `HexGrid` geometry once at the very end.

### The pipeline at a glance

```
HexRegion (coarse)
    │
    ▼
┌─────────────────┐
│ 1. Bounding box  │  O(|region|)
│ 2. Chunk refs    │  O(bbox area / spacing²)
│ 3. Stitch chunks │  O(chunks × chunk_size)  ← numpy block copies
│ 4. Build mapper  │  O(1) closure creation
│ 5. Region filter │  O(merged_size)  ← identifies fine hexes in region
│ 6. Crop proxy    │  O(cropped_size) ← numpy 2D slice
│ 7. Wrap mapper   │  O(1) closure wrapping
│ 8. Build Terrain │  O(cropped_size) ← HexGrid constructed ONCE here
│ 9. Project weather│  O(valid_fine_hexes) ← vectorized numpy scatter
│10. Project sheds │  O(valid_fine_hexes) ← dict grouping + lookup
└─────────────────┘
    │
    ▼
ZoomResult(terrain, basins, mapper, chunks_loaded)
```

### Step 1 — Bounding box with padding

Given a `HexRegion` (a set of coarse hex indices), we find the axis-aligned bounding box in row/col space:

$$r_{\min}, r_{\max}, c_{\min}, c_{\max} = \text{region\_bounding\_box}(\text{region})$$

Then we **pad** by `halo_rings + 1` in every direction (clamped to grid bounds). The padding ensures that when we zoom, fine hexes at the region boundary have enough neighbor data for interpolation and drainage flow:

$$r_{\min}' = \max(0,\; r_{\min} - p) \qquad r_{\max}' = \min(N_r - 1,\; r_{\max} + p)$$

where $p = \text{halo\_rings} + 1$.

**Why halo_rings?** Each chunk already has a halo of extra hexes for seamless stitching. The `+1` ensures we don't get edge artifacts on the region boundary itself.

### Step 2 — Chunk discovery

`bbox_to_chunk_refs` samples the padded bounding box at `spacing = 2 × rings` intervals and maps each sample point to its chunk via `world_to_chunk`:

```
Padded bbox on coarse grid:
┌─────────────────────────┐
│  ·  ·  ·  ·  ·  ·  ·   │   · = sample points at spacing intervals
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   Each · → world_to_chunk → ChunkRef
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   + corners checked explicitly
└─────────────────────────┘
```

The result is a `set[ChunkRef]` — deduplicated because multiple sample points can land in the same chunk.

**Corner check:** The loop at `spacing` intervals can miss chunks that *just barely* overlap a corner. The explicit corner sampling catches those.

### Step 3 — Stitch chunks (the heavy lift)

`stitch_chunks_fast` does:

1. For each `ChunkRef`, look up the cached zoomed chunk as a `DataProxy`
2. On cache miss, call `load_or_generate_chunk` (runs the actual zoom computation), then reload as `DataProxy`
3. Compute the merged grid dimensions from the range of coarse positions × scale
4. Copy each chunk's 2D elevation/watershed arrays into the merged array at its offset

The key equations for merged dimensions:

$$H_{\text{merged}} = (r_{\max}^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale} + H_{\text{chunk}}$$
$$W_{\text{merged}} = (c_{\max}^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale} + W_{\text{chunk}}$$

And the placement offset for each chunk:

$$\text{row\_off}^{(k)} = (r_k^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale}$$
$$\text{col\_off}^{(k)} = (c_k^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale}$$

This is all numpy 2D slice assignment — no per-hex loops.

### Step 4 — Build the mapper (on uncropped proxy)

This is the `build_fine_to_coarse_mapper` from the previous deep dive. The critical point: it's built on the **uncropped** merged proxy, because we need it in Step 5 before we know what to crop.

### Step 5 — Identify the region's fine hexes

This is the step that connects the coarse `HexRegion` to the fine grid. For every valid fine hex:

$$\text{coarse\_idx} = \text{mapper}(\text{fine\_idx})$$

If `coarse_idx ∈ region.hexes`, this fine hex "belongs" to the region.

```
Coarse region (shaded):          Fine merged grid:
┌──┬──┬──┬──┬──┐                 ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │░░│░░│  │   mapper        │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│░░│  │   ←────────     │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│  │  │                 │  │  │  │  │▓▓│▓▓│  │  │  │  │
│  │  │  │  │  │                 │  │  │  │  │  │  │  │  │  │  │
└──┴──┴──┴──┴──┘                 └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

                                  ▓▓ = fine hexes where mapper → region hex
```

Three types of fine hexes get marked **invalid** here:
- Already in `invalidRegion` (from stitching)
- Elevation ≤ −99.0 (deep water sentinel)
- Mapper returns −1 (outside any chunk)

This is the most expensive loop — `O(merged_size)` — but it's pure Python integer ops, no geometry.

### Step 6 — Crop the proxy

The merged proxy can be much larger than the region. Imagine a region that's 5 coarse hexes in a vertical strip — the merged proxy might be 4 chunks wide but we only need a narrow column.

$$\text{crop\_padding} = \max(4,\; \text{scale} \times (\text{halo\_rings} + 1))$$

The crop finds the bounding box of `region_fine_indices`, pads it, and extracts a numpy 2D slice:

```
Before crop (merged proxy):      After crop:
┌────────────────────────┐        ┌──────────┐
│                        │        │          │
│       ┌──────┐         │   →    │ ┌──────┐ │
│       │region│         │        │ │region│ │
│       └──────┘         │        │ └──────┘ │
│                        │        │          │
└────────────────────────┘        └──────────┘
    50×80 = 4000 cells              20×30 = 600 cells
```

**Bail-out:** If cropping would save less than 10% of cells, we skip it (the overhead isn't worth it).

The function returns the crop offsets `(trim_top, trim_left)` needed to translate cropped coordinates back to merged coordinates.

### Step 7 — Wrap the mapper

Since the proxy was cropped, fine indices shifted. `make_cropped_mapper` wraps the original mapper:

$$\text{merged\_row} = \text{cropped\_row} + \text{trim\_top}$$
$$\text{merged\_col} = \text{cropped\_col} + \text{trim\_left}$$
$$\text{merged\_idx} = \text{merged\_row} \times W_{\text{old}} + \text{merged\_col}$$

Then feeds `merged_idx` to the original uncropped mapper. If no cropping happened (trim = 0), the original mapper is returned as-is.

### Step 8 — Build Terrain ONCE

`proxy_to_terrain(cropped_proxy)` is the **only** place where `HexGrid` geometry gets constructed. This is the payoff of the entire DataProxy design:

- `stitch_chunks_fast` → numpy only
- `crop_proxy_to_region` → numpy only
- Steps 5-7 → integer arithmetic only
- **Here** → create hex objects, compute pixel positions, etc.

We do it on the **cropped** proxy, so the HexGrid is as small as possible.

### Step 9 — Project weather (vectorized)

Weather lives on the coarse terrain. We need it on the fine terrain. The mapper gives us the correspondence, and we vectorize it:

```python
fine_arr   = [  0,   1,   2,   5,   6, ...]   # fine indices (valid)
coarse_arr = [ 42,  42,  42,  43,  43, ...]   # their coarse parents
```

For each weather field (temperature, precipitation, etc.):

$$\text{fine\_field}[\text{fine\_arr}] = \text{coarse\_field}[\text{coarse\_arr}]$$

This is a single numpy **fancy-index scatter** — no Python loop over hexes. Every fine hex in a `scale × scale` block gets the same coarse value (nearest-neighbor projection).

**Fallback:** If temperature or precipitation still isn't populated (e.g. the coarse terrain didn't have them), it calls `merged_terrain.compute_weather()` to generate from scratch.

### Step 10 — Project watersheds

We group the fine→coarse mapping into `fine_regions`:

$$\text{fine\_regions}[c] = \{f \;|\; \text{mapper}(f) = c\}$$

This is exactly the dict that `project_watersheds` expects — it assigns each fine hex the watershed ID of its coarse parent from the `DrainageBasins`.

The coarse `DrainageBasins` is lazily computed once (Step 3's `self.basin` check) and then reused for all subsequent zooms.

---

### Worked example

Setup:
- `cover` with `rings=3`, `halo_rings=1`, coarse grid `20×20`
- Region = 6 coarse hexes forming an L-shape
- `scale = 2`

```
Step 1:  bbox = rows [4,8], cols [6,9]
         padded = rows [2,10], cols [4,11]  (padding = 2)

Step 2:  At spacing=6, samples hit chunks (0,0,0) and (1,0,-1)
         → 2 ChunkRefs

Step 3:  Each chunk zooms to 14×14 fine hexes
         Merged proxy = 14×28 = 392 cells  (chunks side by side)

Step 4:  Mapper closure built (captures chunk_offsets, scale=2, etc.)

Step 5:  Loop over 392 fine cells:
         - 168 map to region hexes → region_fine_indices
         - 180 map to non-region hexes → ignored
         - 44 invalid (water/edge)

Step 6:  region_fine_indices bbox = rows [4,17], cols [4,17]
         Crop to 18×18 = 324 cells (saves 17%)
         trim_top=4, trim_left=4

Step 7:  Mapper wrapped: cropped_idx → +4 to both row,col → original mapper

Step 8:  HexGrid(18, 18, radius=5) built — 324 hexes with geometry

Step 9:  temperature[fine_arr] = coarse_temp[coarse_arr]  (one numpy op per field)

Step 10: fine_regions = {42: {0,1,2,3}, 43: {4,5,6,7}, ...}
         Each set gets its parent's watershed_id
         → ZoomResult with 168 meaningful fine hexes
```

### Performance characteristics

| Step | Cost | Dominates when... |
|------|------|-------------------|
| 3. Stitch | O(chunks × chunk_size) | Many chunks, cache cold |
| 5. Region filter | O(merged_size) | Large merged proxy |
| 8. Build Terrain | O(cropped_size) | Large region |
| 9. Weather | O(valid_fine × fields) | Many weather fields |

The cache-cold case (first zoom) is dominated by Step 3 — actually computing the zoomed terrain. Subsequent zooms hit the cache and Steps 5-9 dominate, which are all numpy-fast.

### The returned ZoomResult

```python
@dataclass
class ZoomResult:
    terrain: Terrain          # Full Terrain with HexGrid, elevations, weather fields
    basins: DrainageBasins    # Watersheds projected from coarse → fine
    mapper: Callable          # cropped_fine_idx → coarse_idx (for further lookups)
    chunks_loaded: int        # How many chunks were involved
```

The `mapper` is returned so callers can do their own fine→coarse lookups (e.g., "which biome does this fine hex inherit?" or "what political territory is it in?"). It's the cropped version, so it works directly with indices into `terrain.hexGrid`.

In [ ]:
SVGBuilder.BUILDERHIDE = True

with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server

    print(f"Original: {len(terrain.hexGrid.hexes)} hexes, {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols}")

    result = cover.save(name="Maui Test")
    print(f"Save: {result.status} - {result.context}")
    cover_id = result.id

    loaded = dbg.server.load_cover(cover_id)
    print(f"Load: {loaded.status} - {loaded.context}")

    orig_elev = cover.terrain.elevations
    load_elev = loaded.data.terrain.elevations

    match = np.allclose(orig_elev, load_elev)
    print(f"Elevations match: {match}")

    if match:
        print("✓ Round-trip test passed!")
    else:
        dbg.failed = True
        print("✗ Data mismatch!")

In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Chunk Test")

    # First zoom - should generate
    result1 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"First call: {result1.status} - {result1.context}")

    # Second zoom - should hit cache
    result2 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"Second call: {result2.status} - {result2.context}")

    if result1.status == 'generated' and result2.status == 'cached':
        print("✓ Chunk caching works!")
    else:
        print("✗ Caching issue")

In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    terrain.compute_weather()

    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Full Test")

    basins = cover.zoom_with_full_data(origin=None, scale=2, compute_weather=True)

    print(f"Basin initialized: {cover.basin is not None}")
    print(f"Zoomed terrain: {basins.terrain.hexGrid.nRows}x{basins.terrain.hexGrid.nCols}")
    print(f"Watersheds: {len(basins.sheds)}")

    has_temp = 'temperature' in basins.terrain.fields
    print(f"Has temperature: {has_temp}")

    if cover.basin is not None and len(basins.sheds) > 0 and has_temp:
        print("✓ zoom_with_full_data works!")
    else:
        print("✗ Something missing")